In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json 
from sklearn.preprocessing import StandardScaler
                        

In [ ]:
DATA_PATH = Path("../outputs/freiburg_events.csv") ##file freiburg event i ke dorost kardimo mikhunim



events = pd.read_csv(DATA_PATH)

print(events.shape)
print(events.columns.tolist())

events.head(10)

In [4]:
#events.info() ##data ro negah kon ke kudum numerice va unaro be culumn haye numeric tabdil kon

numeric_columns = [
    "match_id",
    "event_id",
    "time_seconds",
    "player_id",
    "pressure",
    "distance_to_goal",
    "distance_to_opponent",
    "team_pxt",
    "opponent_pxt",
    "minute",
    "plot_x",
    "plot_y",
]

for column in numeric_columns:
    if column in events.columns:
        events[column] = pd.to_numeric(
            events[column],
            errors="coerce",
        )

In [ ]:
## drop kardane event haye gheire mohem: baraye player performance faghat be event hayi niaz darim ke 
## bazikone moshakhas daran

events = events.dropna(
    subset=[
        "match_id",
        "player_id",
        "player_name",
    ]
).copy()

events.head(10)

In [ ]:
##baresi label ha

events["action_type"].value_counts(dropna=False)


In [ ]:
events["action"].value_counts(dropna=False).head(100)


In [ ]:
events["result"].value_counts(dropna=False)


In [ ]:
events["pitch_position"].value_counts(dropna=False)

In [10]:
## sakhte event indicator: 
"""
آیا این Event یک پاس است؟
آیا این Event یک شوت است؟
آیا این Event یک Interception است؟
"""
## 1. involement indicator ##
events["is_pass"] = (
    events["action_type"] == "PASS"
).astype(int)

events["is_reception"] = (
    events["action_type"] == "RECEPTION"
).astype(int)

events["is_dribble"] = (
    events["action_type"] == "DRIBBLE"
).astype(int)

events["is_shot"] = (
    events["action_type"] == "SHOT"
).astype(int)
##با astype(int) مقدارهای Boolean به صفر و یک تبدیل می‌شوند.




## 2. success indicators ##
# pass
events["is_successful_pass"] = (
    (events["action_type"] == "PASS")
    & (events["result"] == "SUCCESS")
).astype(int)

events["is_failed_pass"] = (
    (events["action_type"] == "PASS")
    & (events["result"] == "FAIL")
).astype(int)

# dribble
events["is_successful_dribble"] = (
    (events["action_type"] == "DRIBBLE")
    & (events["result"] == "SUCCESS")
).astype(int)

events["is_failed_dribble"] = (
    (events["action_type"] == "DRIBBLE")
    & (events["result"] == "FAIL")
).astype(int)





## 3. shot indicator ##
"""
در دیتاست تو شوت موفق ظاهراً با SUCCESS نمایش داده می‌شود و احتمالاً متناظر با Goal است.
"""
events["is_successful_shot"] = (
    (events["action_type"] == "SHOT")
    & (events["result"] == "SUCCESS")
).astype(int)




## 4.  Off-Ball indicators 
""" 
Reception actionز:
AVAILABILITY_IN_THE_BACK
AVAILABILITY_OUT_WIDE
AVAILABILITY_BTL
HOLD_UP_PLAY
AVAILABILITY_IN_THE_BOX
AVAILABILITY_FDR
"""

#Between the lines
events["is_between_lines_reception"] = (
    (events["action_type"] == "RECEPTION")
    & (events["action"] == "AVAILABILITY_BTL")
).astype(int)

#Wide availability
events["is_wide_availability"] = (
    (events["action_type"] == "RECEPTION")
    & (events["action"] == "AVAILABILITY_OUT_WIDE")
).astype(int)

#Box availability
events["is_box_availability"] = (
    (events["action_type"] == "RECEPTION")
    & (events["action"] == "AVAILABILITY_IN_THE_BOX")
).astype(int)

#Hold-up play
events["is_hold_up_action"] = (
    (events["action_type"] == "RECEPTION")
    & (events["action"] == "HOLD_UP_PLAY")
).astype(int)

#Availability in the back
events["is_back_availability"] = (
    (events["action_type"] == "RECEPTION")
    & (events["action"] == "AVAILABILITY_IN_THE_BACK")
).astype(int)

#Forward-direction availability
events["is_fdr_availability"] = (
    (events["action_type"] == "RECEPTION")
    & (events["action"] == "AVAILABILITY_FDR")
).astype(int)




## 5. Attacking Zone indicators ###

# Final-third action
events["is_final_third_action"] = (
    events["pitch_position"].isin(
        [
            "FINAL_THIRD",
            "OPPONENT_BOX",
        ]
    )
).astype(int)

#Opponent-box action
events["is_opponent_box_action"] = (
    events["pitch_position"] == "OPPONENT_BOX"
).astype(int)

#Progressive-zone action
events["is_attacking_zone_action"] = (
    events["pitch_position"].isin(
        [
            "FINAL_THIRD",
            "OPPONENT_BOX",
        ]
    )
).astype(int)


## 6. Defensive indicators

events["is_interception"] = (
    events["action_type"] == "INTERCEPTION"
).astype(int)

events["is_loose_ball_regain"] = (
    events["action_type"] == "LOOSE_BALL_REGAIN"
).astype(int)

events["is_ground_duel"] = (
    events["action_type"] == "GROUND_DUEL"
).astype(int)

events["is_clearance"] = (
    events["action_type"] == "CLEARANCE"
).astype(int)

events["is_block"] = (
    events["action_type"] == "BLOCK"
).astype(int)


## 7. Pressure indicators

"""
Pressure عددی بین صفر و ۱۰۰ است
توزیع کلی:

* میانگین حدود ۲۳
* Median حدود ۱۰
* حداکثر ۱۰۰

می‌توانیم یک تعریف اولیه برای High Pressure داشته باشیم:
pressure ehtemalan fesharie ke ye bazikon hengam anjam action tahamol karde
"""

events["is_high_pressure_action"] = (
    events["pressure"] >= 50
).astype(int)

events["is_action_under_high_pressure"] = (
    events["pressure"] >= 50
).astype(int)

In [ ]:
##baressi indicator hayi ke sakhtim baraye player match datasetemun

indicator_columns = [
    "is_pass",
    "is_reception",
    "is_dribble",
    "is_shot",
    "is_successful_pass",
    "is_successful_dribble",
    "is_between_lines_reception",
    "is_wide_availability",
    "is_box_availability",
    "is_hold_up_action",
    "is_back_availability",
    "is_interception",
    "is_loose_ball_regain",
    "is_ground_duel",
    "is_clearance",
    "is_block",
    "is_final_third_action",
    "is_opponent_box_action",
    "is_action_under_high_pressure",
]

events[indicator_columns].sum().sort_values(
    ascending=False
)

In [12]:
## ساخت Player–Match Table : یک بازیکن × یک مسابقه

GROUP_COLUMNS = [
    "match_id",
    "player_id",
    "player_name",
]

#  پوزیشن غالب هر بازیکن در هر مسابقه
def get_mode(series):
    mode = series.mode()

    if mode.empty:
        return np.nan

    return mode.iloc[0]

# Aggregation اصلی
player_match = (
    events
    .groupby(
        GROUP_COLUMNS,
        dropna=False,
    )
    .agg(
        player_position=(
            "player_position",
            get_mode,
        ),

        position_side=(
            "position_side",
            get_mode,
        ),

        first_event_minute=(
            "minute",
            "min",
        ),

        last_event_minute=(
            "minute",
            "max",
        ),

        total_events=(
            "event_id",
            "count",
        ),

        passes=(
            "is_pass",
            "sum",
        ),

        successful_passes=(
            "is_successful_pass",
            "sum",
        ),

        failed_passes=(
            "is_failed_pass",
            "sum",
        ),

        receptions=(
            "is_reception",
            "sum",
        ),

        dribbles=(
            "is_dribble",
            "sum",
        ),

        successful_dribbles=(
            "is_successful_dribble",
            "sum",
        ),

        failed_dribbles=(
            "is_failed_dribble",
            "sum",
        ),

        shots=(
            "is_shot",
            "sum",
        ),

        successful_shots=(
            "is_successful_shot",
            "sum",
        ),

        between_lines_receptions=(
            "is_between_lines_reception",
            "sum",
        ),

        wide_availability=(
            "is_wide_availability",
            "sum",
        ),

        box_availability=(
            "is_box_availability",
            "sum",
        ),

        hold_up_actions=(
            "is_hold_up_action",
            "sum",
        ),

        back_availability=(
            "is_back_availability",
            "sum",
        ),

        fdr_availability=(
            "is_fdr_availability",
            "sum",
        ),

        final_third_actions=(
            "is_final_third_action",
            "sum",
        ),

        opponent_box_actions=(
            "is_opponent_box_action",
            "sum",
        ),

        interceptions=(
            "is_interception",
            "sum",
        ),

        loose_ball_regains=(
            "is_loose_ball_regain",
            "sum",
        ),

        ground_duels=(
            "is_ground_duel",
            "sum",
        ),

        clearances=(
            "is_clearance",
            "sum",
        ),

        blocks=(
            "is_block",
            "sum",
        ),

        high_pressure_actions=(
            "is_action_under_high_pressure",
            "sum",
        ),

        average_pressure=(
            "pressure",
            "mean",
        ),

        total_pxt=(
            "team_pxt",
            "sum",
        ),

        average_pxt=(
            "team_pxt",
            "mean",
        ),

        max_pxt=(
            "team_pxt",
            "max",
        ),

        average_distance_to_goal=(
            "distance_to_goal",
            "mean",
        ),

        average_x=(
            "plot_x",
            "mean",
        ),

        average_y=(
            "plot_y",
            "mean",
        ),
    )
    .reset_index()
)

In [ ]:
## baresi player match table i ke sakhtim

print(player_match.shape)

player_match.head(20)

In [ ]:
""""
بخش ۱۴ — Minutes Played

این مهم‌ترین داده‌ای است که هنوز باید اضافه شود.

چرا Minutes لازم است؟

فرض کن:

* بازیکن A در ۹۰ دقیقه ۲۰ پاس داده
* بازیکن B در ۱۵ دقیقه ۱۰ پاس داده

Count خام می‌گوید A بیشتر پاس داده، اما نرخ مشارکت B بالاتر است.

بنابراین باید Countها را Per 90 کنیم.

بهترین روش

یک DataFrame جدا با این ساختار داشته باش:
match_id | player_id | minutes
"""



In [15]:
## defining minute_df from players KPI
PLAYER_KPIS_DIR = Path(
    "../data/player_kpis"
)
FREIBURG_SQUAD_ID = 34

player_kpi_files = sorted(

    PLAYER_KPIS_DIR.glob("*.json")

)

## function estekhraje minutes az file

def extract_freiburg_minutes_from_kpi_file(
    file_path: Path,
    freiburg_squad_id: int = 34,
) -> list[dict]:
    with open(
        file_path,
        "r",
        encoding="utf-8",
    ) as file:
        data = json.load(file)


    match_id = data.get("matchId")

    rows = []

    for squad_key in [
        "squadHome",
        "squadAway",
    ]:
        squad = data.get(squad_key, {})

        squad_id = squad.get("id")

        if squad_id != freiburg_squad_id:
            continue

        for player in squad.get(
            "players",
            [],
        ):
            play_duration_seconds = (
                player.get("playDuration")
            )

            if play_duration_seconds is None:
                minutes = np.nan
            else:
                minutes = (
                    play_duration_seconds
                    / 60
                )

            rows.append(
                {
                    "match_id": match_id,
                    "player_id": player.get("id"),
                    "minutes": minutes,
                    "play_duration_seconds":
                        play_duration_seconds,
                    "match_share":
                        player.get("matchShare"),
                    "kpi_position":
                        player.get("position"),
                    "squad_id": squad_id,
                }
            )

    return rows


In [16]:
### sakhte minute_df az tamame file haye player kpi


all_minutes_rows = []

for file_path in player_kpi_files:
    file_rows = (
        extract_freiburg_minutes_from_kpi_file(
            file_path=file_path,
            freiburg_squad_id=(
                FREIBURG_SQUAD_ID
            ),
        )
    )

    all_minutes_rows.extend(
        file_rows
    )

minutes_full_df = pd.DataFrame(
    all_minutes_rows
)

In [ ]:
##barresi
print(minutes_full_df.shape)

minutes_full_df.head()

In [ ]:
## برای کنترل خودکار:

match_duration_check = (
    minutes_full_df[
        "play_duration_seconds"
    ]
    / minutes_full_df[
        "match_share"
    ].replace(0, np.nan)
)

minutes_full_df[
    "estimated_match_duration_seconds"
] = match_duration_check

minutes_full_df[
    [
        "match_id",
        "player_id",
        "play_duration_seconds",
        "match_share",
        "estimated_match_duration_seconds",
    ]
].head(20)

In [19]:
##  پاک‌سازی انواع داده

numeric_columns = [
    "match_id",
    "player_id",
    "minutes",
    "play_duration_seconds",
    "match_share",
    "squad_id",
]

# حذف ردیف‌های ناقص:
for column in numeric_columns:
    minutes_full_df[column] = (
        pd.to_numeric(
            minutes_full_df[column],
            errors="coerce",
        )
    )

    minutes_full_df = (
    minutes_full_df
    .dropna(
        subset=[
            "match_id",
            "player_id",
            "minutes",
        ]
    )
    .copy()
)
    
## تبدیل IDها:
minutes_full_df["match_id"] = (
    minutes_full_df[
        "match_id"
    ].astype(int)
)

minutes_full_df["player_id"] = (
    minutes_full_df[
        "player_id"
    ].astype(int)
)

## hazfe maghadire gheyre manteghi az nazare zaman baraye bazi

minutes_full_df = minutes_full_df[
    minutes_full_df["minutes"].between(
        0,
        130,
        inclusive="both",
    )
].copy()

In [ ]:
## final version minute_df

minutes_df = (
    minutes_full_df[
        [
            "match_id",
            "player_id",
            "minutes",
        ]
    ]
    .copy()
)

minutes_df.head(20)

## barresie duplicate haye
duplicate_minutes = minutes_df[
    minutes_df.duplicated(
        subset=[
            "match_id",
            "player_id",
        ],
        keep=False,
    )
].sort_values(
    [
        "match_id",
        "player_id",
    ]
)

duplicate_minutes

In [21]:


## merge minute_df ba player_match

player_match["match_id"] = (
    pd.to_numeric(
        player_match["match_id"],
        errors="coerce",
    ).astype("Int64")
)

player_match["player_id"] = (
    pd.to_numeric(
        player_match["player_id"],
        errors="coerce",
    ).astype("Int64")
)

minutes_df["match_id"] = (
    minutes_df["match_id"]
    .astype("Int64")
)

minutes_df["player_id"] = (
    minutes_df["player_id"]
    .astype("Int64")
)

## merge 

player_match = player_match.merge(
    minutes_df,
    on=[
        "match_id",
        "player_id",
    ],
    how="left",
    validate="one_to_many",
)

In [ ]:
## barresie missing minutes
missing_minutes_count = (
    player_match[
        "minutes"
    ]
    .isna()
    .sum()
)

print(
    "Missing minutes:",
    missing_minutes_count,
)


In [23]:
## saving the files

minutes_full_df.to_csv(
    "../outputs/"
    "player_match_minutes_full.csv",
    index=False,
)

minutes_df.to_csv(
    "../outputs/"
    "player_match_minutes.csv",
    index=False,
)

In [ ]:
## bakhshe 15  فیلتر حداقل زمان بازی

player_match["minutes"] = pd.to_numeric(
    player_match["minutes"],
    errors="coerce",
)
player_match["minutes"].describe()


In [25]:
## hazfe player_match haye kamtar az 30 min
##چرا ۳۰ دقیقه؟
#چون Rateهای Per 90 برای بازیکنی که فقط ۵ یا ۱۰ دقیقه بازی کرده بسیار ناپایدار می‌شوند.

MIN_MINUTES = 30

player_match_filtered = player_match[
    player_match["minutes"] >= MIN_MINUTES
].copy()

In [26]:
## ساخت Per-90 Features:
# formula : Per 90 = Count × 90 / Minutes Played

def calculate_per90(
    dataframe,
    columns,
    minutes_column="minutes",
):
    result = dataframe.copy()

    for column in columns:
        result[f"{column}_per90"] = (
            result[column]
            * 90
            / result[minutes_column]
        )

    return result


## Featureهای Count:

COUNT_FEATURES = [
    "total_events",
    "passes",
    "receptions",
    "dribbles",
    "shots",
    "between_lines_receptions",
    "wide_availability",
    "box_availability",
    "hold_up_actions",
    "back_availability",
    "fdr_availability",
    "final_third_actions",
    "opponent_box_actions",
    "interceptions",
    "loose_ball_regains",
    "ground_duels",
    "clearances",
    "blocks",
    "high_pressure_actions",
    "total_pxt",
]

player_match_filtered = calculate_per90(
    player_match_filtered,
    COUNT_FEATURES,
)



In [27]:
### ساخت Success Rate

#Pass Success Rate
player_match_filtered[
    "pass_attempts_with_result"
] = (
    player_match_filtered["successful_passes"]
    + player_match_filtered["failed_passes"]
)

player_match_filtered[
    "pass_success_rate"
] = np.where(
    player_match_filtered[
        "pass_attempts_with_result"
    ] > 0,

    player_match_filtered[
        "successful_passes"
    ]
    / player_match_filtered[
        "pass_attempts_with_result"
    ],

    np.nan,
)


#Dribble Success Rate
player_match_filtered[
    "dribble_attempts_with_result"
] = (
    player_match_filtered["successful_dribbles"]
    + player_match_filtered["failed_dribbles"]
)
player_match_filtered[
    "dribble_success_rate"
] = np.where(
    player_match_filtered[
        "dribble_attempts_with_result"
    ] > 0,

    player_match_filtered[
        "successful_dribbles"
    ]
    / player_match_filtered[
        "dribble_attempts_with_result"
    ],

    np.nan,
)

In [28]:
## جلوگیری از Rateهای غیرقابل‌اعتماد
"""
یک بازیکن ممکن است فقط یک Dribble داشته باشد
 و موفق شده باشد؛ Rate او ۱۰۰٪ می‌شود، ولی این خیلی قابل‌اعتماد نیست.

Dribble Success Rate را فقط وقتی نگه داریم که حداقل دو Attempt وجود داشته باشد:
"""

# for dribble
player_match_filtered.loc[
    player_match_filtered[
        "dribble_attempts_with_result"
    ] < 2,
    "dribble_success_rate",
] = np.nan

# for pass
player_match_filtered.loc[
    player_match_filtered[
        "pass_attempts_with_result"
    ] < 5,
    "pass_success_rate",
] = np.nan

In [29]:
## تعریف Featureهای Performance نهایی
# نسخه اولیه Feature Matrix:

CLUSTER_FEATURES = [
    # Involvement
    "passes_per90",
    "receptions_per90",
    "dribbles_per90",

    # Execution quality
    "pass_success_rate",
    "dribble_success_rate",

    # Attacking threat
    "shots_per90",
    "total_pxt_per90",
    "final_third_actions_per90",
    "opponent_box_actions_per90",

    # Off-ball performance
    "between_lines_receptions_per90",
    "wide_availability_per90",
    "box_availability_per90",
    "hold_up_actions_per90",

    # Defensive contribution
    "interceptions_per90",
    "loose_ball_regains_per90",
    "ground_duels_per90",
    "clearances_per90",
    "blocks_per90",

    # Playing under pressure
    "high_pressure_actions_per90",
]

In [ ]:
### barresie missing value
# اگر یک Feature بیش از ۴۰ یا ۵۰ درصد Missing داشت، باید درباره حذفش فکر کنیم.
missing_report = (
    player_match_filtered[
        CLUSTER_FEATURES
    ]
    .isna()
    .mean()
    .sort_values(
        ascending=False
    )
)

missing_report

In [31]:
### حذف Featureهای بدون Variance

unique_counts = (
    player_match_filtered[
        CLUSTER_FEATURES
    ]
    .nunique()
    .sort_values()
)

unique_counts

# هر Feature که فقط یک مقدار داشته باشد:
constant_features = unique_counts[
    unique_counts <= 1
].index.tolist()

constant_features
# آن‌ها را حذف کن:
CLUSTER_FEATURES = [
    feature
    for feature in CLUSTER_FEATURES
    if feature not in constant_features
]

In [ ]:
## بررسی توزیع Featureها

player_match_filtered[
    CLUSTER_FEATURES
].describe().T

player_match_filtered[
    CLUSTER_FEATURES
].hist(
    figsize=(16, 14),
    bins=20,
)

plt.tight_layout()
plt.show()

In [33]:
## کنترل Outlierها
# برای K-Means، Outlierها خطرناک‌اند چون K-Means براساس فاصله کار می‌کند.
# یک نسخه جدا برای مدل بساز:

model_data = player_match_filtered[
    CLUSTER_FEATURES
].copy()

for column in CLUSTER_FEATURES:
    lower_bound = model_data[
        column
    ].quantile(0.01)

    upper_bound = model_data[
        column
    ].quantile(0.99)

    model_data[column] = (
        model_data[column]
        .clip(
            lower=lower_bound,
            upper=upper_bound,
        )
    )

    ## این کار ردیف‌ها را حذف نمی‌کند؛ فقط مقادیر شدید را محدود می‌کند.

In [ ]:
## پر کردن Missing Values

feature_medians = model_data.median(
    numeric_only=True
)

model_data = model_data.fillna(
    feature_medians
)
print(
    model_data.isna().sum().sum()
)

In [ ]:
## بررسی همبستگی Featureها
correlation_matrix = model_data.corr()

# نمایش:
plt.figure(figsize=(15, 12))

plt.imshow(
    correlation_matrix,
    aspect="auto",
)

plt.colorbar()

plt.xticks(
    range(len(CLUSTER_FEATURES)),
    CLUSTER_FEATURES,
    rotation=90,
)

plt.yticks(
    range(len(CLUSTER_FEATURES)),
    CLUSTER_FEATURES,
)

plt.title(
    "Correlation Matrix of Performance Features"
)

plt.tight_layout()
plt.show()

In [36]:
features_to_drop = [
    "receptions_per90",
    "dribbles_per90",
]

In [37]:
CLUSTER_FEATURES = [
    feature
    for feature in CLUSTER_FEATURES
    if feature not in features_to_drop
]

In [38]:
model_data = player_match_filtered[
    CLUSTER_FEATURES
].copy()

In [39]:
for column in CLUSTER_FEATURES:
    lower = model_data[column].quantile(0.01)
    upper = model_data[column].quantile(0.99)

    model_data[column] = model_data[column].clip(
        lower=lower,
        upper=upper,
    )

model_data = model_data.fillna(
    model_data.median()
)

In [ ]:
## بررسی اینکه Clusterها فقط پست را یاد نگیرند
""""
چون بعضی Featureها شدیداً وابسته به پست هستند، باید توزیع Featureها را براساس Position بررسی کنیم.
"""

position_summary = (
    player_match_filtered
    .groupby("player_position")[
        CLUSTER_FEATURES
    ]
    .mean()
    .round(2)
)

position_summary

In [41]:
## A — Goalkeeper را حذف کنیم
# چون Performance Featureهای دروازه‌بان کاملاً با بازیکنان زمین متفاوت است.

player_match_filtered = (
    player_match_filtered[
        player_match_filtered[
            "player_position"
        ] != "GOALKEEPER"
    ]
    .copy()
)

In [42]:
model_data = player_match_filtered[
    CLUSTER_FEATURES
].copy()

In [43]:
for column in CLUSTER_FEATURES:
    lower = model_data[column].quantile(0.01)
    upper = model_data[column].quantile(0.99)

    model_data[column] = (
        model_data[column]
        .clip(lower, upper)
    )
    ## دوباره Outlier clipping:

In [44]:
model_data = model_data.fillna(
    model_data.median()
)
## دوباره Median Imputation:

In [45]:
## ذخیره Player–Match Feature Table
OUTPUT_PATH = Path(
    "../outputs/player_match_features.csv"
)

player_match_filtered.to_csv(
    OUTPUT_PATH,
    index=False,
)

In [ ]:
## sakhte feature matrix nahayi
X = model_data.copy()
print("Number of samples:", X.shape[0])
print("Number of features:", X.shape[1])

X.head()

## har radif یک Player–Match Performance
## har column یک Performance Feature

In [ ]:
### بخش ۳۱ — StandardScaler

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

X_scaled_df = pd.DataFrame(
    X_scaled,
    columns=CLUSTER_FEATURES,
    index=X.index,
)

scaling_check = pd.DataFrame(
    {
        "mean": X_scaled_df.mean(),
        "std": X_scaled_df.std(),
    }
)

scaling_check.round(2)

In [ ]:
player_match_filtered.shape

In [ ]:
player_match_filtered[
    CLUSTER_FEATURES
].describe().T

In [ ]:
missing_report

In [ ]:
player_match_filtered[
    "player_position"
].value_counts()

In [ ]:
CLUSTER_FEATURES = [
    # Involvement
    "passes_per90",

    # Execution quality
    "pass_success_rate",
    "dribble_success_rate",

    # Attacking threat
    "shots_per90",
    "total_pxt_per90",
    "final_third_actions_per90",
    "opponent_box_actions_per90",

    # Off-ball performance
    "between_lines_receptions_per90",
    "wide_availability_per90",
    "box_availability_per90",
    "hold_up_actions_per90",

    # Defensive contribution
    "interceptions_per90",
    "loose_ball_regains_per90",
    "ground_duels_per90",
    "clearances_per90",
    "blocks_per90",

    # Actions performed under pressure
    "high_pressure_actions_per90",
]
print(len(CLUSTER_FEATURES))
print(CLUSTER_FEATURES)


In [ ]:
## model o dobare misazim bade emale taghirat

model_data = player_match_filtered[
    CLUSTER_FEATURES
].copy()

print(model_data.shape)

In [ ]:
##Median imputation , فقط یک مقدار Missing داری، اما باید آن را قبل از scaler و K-Means پر کنیم.

feature_medians = model_data.median()

model_data = model_data.fillna( ## por kardan missing value ba median
    feature_medians
)
print( ## missing value bayad sefr bashe
    "Missing values:",
    model_data.isna().sum().sum()
)

In [55]:
model_data_raw = model_data.copy() ## noskheye asli model

In [56]:
## noskheye clipped shode bekhatere outliers ha
model_data_clipped = model_data.copy()

clipping_bounds = {}

for column in CLUSTER_FEATURES:
    lower_bound = model_data_clipped[
        column
    ].quantile(0.01)

    upper_bound = model_data_clipped[
        column
    ].quantile(0.99)

    clipping_bounds[column] = {
        "lower": lower_bound,
        "upper": upper_bound,
    }

    model_data_clipped[column] = (
        model_data_clipped[column]
        .clip(
            lower=lower_bound,
            upper=upper_bound,
        )
    )
    model_data_clipped.describe().T

In [ ]:
## dobare correlation o mohasebe mikonim
correlation_matrix = (
    model_data_clipped.corr()
)

high_correlations = []

for i in range(
    len(correlation_matrix.columns)
):
    for j in range(i):
        correlation = (
            correlation_matrix.iloc[i, j]
        )

        if abs(correlation) >= 0.85:
            high_correlations.append(
                {
                    "feature_1":
                        correlation_matrix.columns[i],

                    "feature_2":
                        correlation_matrix.columns[j],

                    "correlation":
                        correlation,
                }
            )

## barresie joz haye correlation dar
high_correlation_df = pd.DataFrame(
    high_correlations,
    columns=[
        "feature_1",
        "feature_2",
        "correlation",
    ],
)

if not high_correlation_df.empty:
    high_correlation_df = (
        high_correlation_df
        .sort_values(
            "correlation",
            ascending=False,
        )
        .reset_index(drop=True)
    )

high_correlation_df 

In [ ]:
## barresie feature haye kam etelaat

zero_percentage = (
    model_data_clipped
    .eq(0)
    .mean()
    .sort_values(
        ascending=False
    )
)

zero_percentage

In [ ]:
zero_report = pd.DataFrame(
    {
        "zero_percentage":
            zero_percentage,

        "non_zero_samples":
            model_data_clipped
            .ne(0)
            .sum(),
    }
)

zero_report

In [60]:
## sakhte scalar dobare
scaler = StandardScaler()
X_scaled = scaler.fit_transform(
    model_data_clipped
)
X_scaled_df = pd.DataFrame(
    X_scaled,
    columns=CLUSTER_FEATURES,
    index=model_data_clipped.index,
)

In [ ]:
## barresi
scaling_check = pd.DataFrame(
    {
        "mean":
            X_scaled_df.mean(),

        "std":
            X_scaled_df.std(ddof=0),
    }
)

scaling_check.round(3)

In [62]:
## ذخیره Metadata هم‌تراز با X
#برای اینکه بعد از clustering بدانیم هر ردیف متعلق به چه بازیکن و مسابقه‌ای است:

METADATA_COLUMNS = [
    "match_id",
    "player_id",
    "player_name",
    "player_position",
    "position_side",
    "minutes",
]

metadata_df = (
    player_match_filtered[
        METADATA_COLUMNS
    ]
    .loc[
        X_scaled_df.index
    ]
    .copy()
)

## alignment
assert metadata_df.index.equals(
    X_scaled_df.index
)

In [ ]:
print(X_scaled_df)
X_scaled_df.to_csv(Path(
    "../outputs/X_scaled_df.csv"
))

print(CLUSTER_FEATURES)

In [64]:
##PCA برای تحلیل و Visualization
""""
1. PCA را برای بررسی ساختار داده اجرا کنیم.
2. Explained variance را ببینیم.
3. برای visualization از دو Component استفاده کنیم.
4. ابتدا K-Means را روی تمام Featureهای Scale‌شده اجرا کنیم.
5. بعد نتیجه را با نسخه PCA-based مقایسه کنیم.
"""

from sklearn.decomposition import PCA
import pandas as pd 
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

In [65]:
## PCA kamel
pca_full = PCA()

X_pca_full = pca_full.fit_transform(
    X_scaled_df
)

In [ ]:
## Explained variance:

explained_variance = (
    pca_full.explained_variance_ratio_
)

cumulative_variance = np.cumsum(
    explained_variance
)

#جدول:
pca_variance_df = pd.DataFrame(
    {
        "component":
            np.arange(
                1,
                len(explained_variance) + 1,
            ),

        "explained_variance":
            explained_variance,

        "cumulative_variance":
            cumulative_variance,
    }
)

pca_variance_df

In [ ]:
#رسم Cumulative Explained Variance

plt.figure(figsize=(9, 5))

plt.plot(
    pca_variance_df["component"],
    pca_variance_df[
        "cumulative_variance"
    ],
    marker="o",
)

plt.axhline(
    y=0.80,
    linestyle="--",
    label="80% variance",
)

plt.axhline(
    y=0.90,
    linestyle="--",
    label="90% variance",
)

plt.xlabel(
    "Number of Principal Components"
)

plt.ylabel(
    "Cumulative Explained Variance"
)

plt.title(
    "PCA Cumulative Explained Variance"
)

plt.xticks(
    pca_variance_df["component"]
)

plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# تعداد Component لازم برای ۸۰٪:
# 
n_components_80 = (
    np.argmax(
        cumulative_variance >= 0.80
    )
    + 1
)

print(
    "Components for 80% variance:",
    n_components_80,
)

#برای ۹۰٪:
n_components_90 = (
   np.argmax(
     cumulative_variance >= 0.90
     )
    + 1
)

print(
    "Components for 90% variance:",
    n_components_90,
)


In [ ]:
## PCA دو بعدی برای Visualization

pca_2d = PCA(
    n_components=2
)

X_pca_2d = pca_2d.fit_transform(
    X_scaled_df
)

##Dataframe
pca_2d_df = pd.DataFrame(
    X_pca_2d,
    columns=[
        "pca_1",
        "pca_2",
    ],
    index=X_scaled_df.index,
)

## میزان اطلاعات دو Component:

print(
    "PCA 1 variance:",
    pca_2d.explained_variance_ratio_[0],
)

print(
    "PCA 2 variance:",
    pca_2d.explained_variance_ratio_[1],
)

print(
    "Total 2D variance:",
    pca_2d.explained_variance_ratio_.sum(),
)

In [70]:
## بررسی PCA Loadings

pca_loadings = pd.DataFrame(
    pca_2d.components_.T,
    index=CLUSTER_FEATURES,
    columns=[
        "PC1",
        "PC2",
    ],
)

In [71]:
## مرتب‌سازی براساس اثر بر PC1:
pca_loadings[
    "abs_PC1"
] = pca_loadings[
    "PC1"
].abs()

pca_loadings[
    "abs_PC2"
] = pca_loadings[
    "PC2"
].abs()



In [ ]:
## بیشترین Featureهای PC1:

pca_loadings.sort_values(
    "abs_PC1",
    ascending=False,
).head(10)

##بیشترین Featureهای PC2:
pca_loadings.sort_values(
    "abs_PC2",
    ascending=False,
).head(10)


In [73]:
## آماده‌سازی ارزیابی K-Means

from sklearn.cluster import KMeans

from sklearn.metrics import (
    silhouette_score,
    calinski_harabasz_score,
    davies_bouldin_score,
)
#قرار نیست فقط یک k را حدس بزنیم.

In [ ]:
## ارزیابی تعداد Clusterها
k_results = []

for k in range(2, 9):
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=50,
        max_iter=500,
    )

    cluster_labels = (
        kmeans.fit_predict(
            X_scaled_df
        )
    )

    k_results.append(
        {
            "k": k,

            "inertia":
                kmeans.inertia_,

            "silhouette":
                silhouette_score(
                    X_scaled_df,
                    cluster_labels,
                ),

            "calinski_harabasz":
                calinski_harabasz_score(
                    X_scaled_df,
                    cluster_labels,
                ),

            "davies_bouldin":
                davies_bouldin_score(
                    X_scaled_df,
                    cluster_labels,
                ),

            "smallest_cluster":
                pd.Series(
                    cluster_labels
                )
                .value_counts()
                .min(),

            "largest_cluster":
                pd.Series(
                    cluster_labels
                )
                .value_counts()
                .max(),
        }
    )

## natije:
k_evaluation_df = pd.DataFrame(
k_results
)

k_evaluation_df

In [ ]:
##رسم Elbow Plot
plt.figure(figsize=(8, 5))

plt.plot(
    k_evaluation_df["k"],
    k_evaluation_df["inertia"],
    marker="o",
)

plt.xlabel("Number of clusters (k)")
plt.ylabel("Inertia")
plt.title("K-Means Elbow Method")
plt.xticks(k_evaluation_df["k"])
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
## رسم Silhouette Score

plt.figure(figsize=(8, 5))

plt.plot(
    k_evaluation_df["k"],
    k_evaluation_df["silhouette"],
    marker="o",
)

plt.xlabel("Number of clusters (k)")
plt.ylabel("Silhouette Score")
plt.title("Silhouette Score by Number of Clusters")
plt.xticks(k_evaluation_df["k"])
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [77]:
## ساخت و مقایسه مدل‌های کاندید

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
## سه مدل را Fit کن:
CANDIDATE_K = [2, 3, 5]

candidate_models = {}
candidate_labels = {}

for k in CANDIDATE_K:
    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=100,
        max_iter=500,
    )

    labels = model.fit_predict(
        X_scaled_df
    )

    candidate_models[k] = model
    candidate_labels[k] = labels

    print(
        f"k={k} | "
        f"silhouette="
        f"{silhouette_score(X_scaled_df, labels):.4f}"
    )

In [ ]:
## برای هر k، اندازه خوشه‌ها را ببین:

cluster_size_comparison = []

for k, labels in candidate_labels.items():
    sizes = (
        pd.Series(labels)
        .value_counts()
        .sort_index()
    )

    for cluster_id, count in sizes.items():
        cluster_size_comparison.append(
            {
                "k": k,
                "cluster": cluster_id,
                "count": count,
                "percentage": (
                    count / len(labels) * 100
                ),
            }
        )

cluster_size_comparison_df = pd.DataFrame(
    cluster_size_comparison
)

cluster_size_comparison_df

In [ ]:
##برای نمایش Pivot:
cluster_size_pivot = (
    cluster_size_comparison_df
    .pivot(
        index="cluster",
        columns="k",
        values="count",
    )
)

cluster_size_pivot

In [ ]:
cluster_silhouette_rows = []

for k, labels in candidate_labels.items():
    sample_scores = silhouette_samples(
        X_scaled_df,
        labels,
    )

    for cluster_id in sorted(
        np.unique(labels)
    ):
        mask = labels == cluster_id

        cluster_silhouette_rows.append(
            {
                "k": k,
                "cluster": cluster_id,
                "size": int(mask.sum()),
                "mean_silhouette":
                    sample_scores[mask].mean(),
                "median_silhouette":
                    np.median(
                        sample_scores[mask]
                    ),
                "negative_percentage":
                    (
                        sample_scores[mask] < 0
                    ).mean() * 100,
            }
        )

cluster_silhouette_df = pd.DataFrame(
    cluster_silhouette_rows
)

cluster_silhouette_df.round(3)

In [82]:
candidate_scaled_profiles = {}

for k, labels in candidate_labels.items():
    profile = (
        X_scaled_df
        .assign(cluster=labels)
        .groupby("cluster")
        .mean()
    )

    candidate_scaled_profiles[k] = profile

In [ ]:
candidate_scaled_profiles[2].T

In [ ]:
candidate_scaled_profiles[3].T

In [ ]:
candidate_scaled_profiles[5].T

In [86]:
def summarize_cluster_profiles(
    profile_df,
    top_n=5,
):
    summaries = []

    for cluster_id in profile_df.index:
        profile = (
            profile_df
            .loc[cluster_id]
            .sort_values(
                ascending=False
            )
        )

        summaries.append(
            {
                "cluster": cluster_id,
                "strongest_features":
                    profile.head(top_n),
                "weakest_features":
                    profile.tail(top_n),
            }
        )

    return summaries

In [ ]:
summary_k2 = summarize_cluster_profiles(
    candidate_scaled_profiles[2]
)

for result in summary_k2:
    print(
        f"\nCluster {result['cluster']}"
    )

    print("\nStrongest:")
    print(result["strongest_features"])

    print("\nWeakest:")
    print(result["weakest_features"])

In [ ]:
summary_k3 = summarize_cluster_profiles(
    candidate_scaled_profiles[3]
)

for result in summary_k3:
    print(
        f"\nCluster {result['cluster']}"
    )

    print("\nStrongest:")
    print(result["strongest_features"])

    print("\nWeakest:")
    print(result["weakest_features"])

In [ ]:
candidate_position_tables = {}

for k, labels in candidate_labels.items():
    temp_df = metadata_df.copy()

    temp_df["cluster"] = labels

    position_table = pd.crosstab(
        temp_df["cluster"],
        temp_df["player_position"],
        normalize="index",
    )

    candidate_position_tables[k] = (
        position_table
    )

#candidate_position_tables[2].round(2)
candidate_position_tables[3].round(2)
#candidate_position_tables[5].round(2)

In [90]:
candidate_player_variation = {}

for k, labels in candidate_labels.items():
    temp_df = metadata_df.copy()

    temp_df["cluster"] = labels

    variation = (
        temp_df
        .groupby("player_name")[
            "cluster"
        ]
        .nunique()
        .sort_values(
            ascending=False
        )
    )

    candidate_player_variation[k] = (
        variation
    )

In [ ]:
candidate_player_variation[3]

In [ ]:
for k, variation in (
    candidate_player_variation.items()
):
    multi_cluster_rate = (
        variation.gt(1).mean()
    )

    print(
        f"k={k}: "
        f"{multi_cluster_rate:.1%} "
        f"of players appear in "
        f"multiple clusters"
    )

In [93]:
from sklearn.metrics import (
    adjusted_rand_score,
)

def evaluate_kmeans_stability(
    X,
    k,
    seeds=range(10),
):
    label_sets = []

    for seed in seeds:
        model = KMeans(
            n_clusters=k,
            random_state=seed,
            n_init=50,
            max_iter=500,
        )

        labels = model.fit_predict(X)

        label_sets.append(labels)

    ari_scores = []

    for i in range(len(label_sets)):
        for j in range(i):
            ari = adjusted_rand_score(
                label_sets[i],
                label_sets[j],
            )

            ari_scores.append(ari)

    return {
        "k": k,
        "mean_ari": np.mean(ari_scores),
        "min_ari": np.min(ari_scores),
        "max_ari": np.max(ari_scores),
    }

In [ ]:
stability_results = []

for k in CANDIDATE_K:
    result = evaluate_kmeans_stability(
        X_scaled_df,
        k,
    )

    stability_results.append(result)

stability_df = pd.DataFrame(
    stability_results
)

stability_df.round(3)

In [ ]:
### stage e bad az barresi va entekhab k = 3
from sklearn.cluster import KMeans

FINAL_K = 3
RANDOM_STATE = 42
print("Final number of clusters:", FINAL_K)
print("Random state:", RANDOM_STATE)

# model nahayi
final_kmeans = KMeans(
    n_clusters=FINAL_K,
    random_state=RANDOM_STATE,
    n_init=200,
    max_iter=500,
)

In [ ]:
## daryafte label ha
cluster_labels = final_kmeans.fit_predict(
    X_scaled_df
)

print("Number of labels:", len(cluster_labels))
print("Unique clusters:", sorted(set(cluster_labels)))

In [ ]:
## mohasebeye meyare nahayie model

from sklearn.metrics import (
    silhouette_score,
    calinski_harabasz_score,
    davies_bouldin_score,
)
final_silhouette = silhouette_score(
    X_scaled_df,
    cluster_labels,
)
final_calinski_harabasz = (
    calinski_harabasz_score(
        X_scaled_df,
        cluster_labels,
    )
)
final_davies_bouldin = (
    davies_bouldin_score(
        X_scaled_df,
        cluster_labels,
    )
)
print(f"Silhouette score: " f"{final_silhouette:.4f}")
print(f"Calinski-Harabasz score: "f"{final_calinski_harabasz:.4f}")
print(f"Davies-Bouldin score: "f"{final_davies_bouldin:.4f}")

In [ ]:
## ساخت جدول نهایی Player–Match Clustered

player_match_clustered = (
    player_match_filtered
    .loc[X_scaled_df.index]
    .copy()
)

## اضافه‌کردن Cluster:
player_match_clustered[
    "cluster"
] = cluster_labels

assert len(
    player_match_clustered
) == len(cluster_labels)

assert (
    player_match_clustered.index
    .equals(X_scaled_df.index)
)

print(player_match_clustered.shape)

In [ ]:
###مرحله ۵ — اضافه‌کردن مختصات PCA
# از pca_2d_df که قبلاً ساخته‌ای استفاده کن:

from sklearn.decomposition import PCA

pca_2d = PCA(n_components=2)
X_pca_2d = pca_2d.fit_transform(X_scaled_df)

pca_2d_df = pd.DataFrame(
    X_pca_2d,
    columns=["pca_1", "pca_2"],
    index=X_scaled_df.index,
)

player_match_clustered[
    "pca_1"
] = pca_2d_df.loc[
    player_match_clustered.index,
    "pca_1",
]

player_match_clustered[
    "pca_2"
] = pca_2d_df.loc[
    player_match_clustered.index,
    "pca_2",
]

player_match_clustered[
    [
        "match_id",
        "player_name",
        "cluster",
        "pca_1",
        "pca_2",
    ]
].head(20)


In [ ]:
## بررسی اندازه Clusterها

cluster_sizes = (player_match_clustered["cluster"].value_counts().sort_index())

cluster_size_report = pd.DataFrame(
    {"count": cluster_sizes, "percentage": (cluster_sizes / cluster_sizes.sum() * 100)}
)
cluster_size_report.round(2)

In [ ]:
## Silhouette هر Cluster 
from sklearn.metrics import silhouette_samples
sample_silhouette_scores = (
    silhouette_samples(
        X_scaled_df,
        cluster_labels,
    )
)

player_match_clustered[
    "silhouette_value"
] = sample_silhouette_scores

## gozareshe har cluster
cluster_silhouette_report = (
    player_match_clustered
    .groupby("cluster")
    ["silhouette_value"]
    .agg(
        size="count",
        mean_silhouette="mean",
        median_silhouette="median",
        minimum_silhouette="min",
        maximum_silhouette="max",
    )
)

negative_silhouette_percentage = (
    player_match_clustered
    .assign(
        negative_silhouette=(
            player_match_clustered[
                "silhouette_value"
            ] < 0
        )
    )
    .groupby("cluster")
    ["negative_silhouette"]
    .mean()
    * 100
)

## adding to report
cluster_silhouette_report[
    "negative_percentage"
] = negative_silhouette_percentage

cluster_silhouette_report.round(3)

In [ ]:
## ساخت پروفایل استانداردشده Clusterها
# مهم‌ترین جدول برای مقایسه سه Cluster:

cluster_profile_scaled = (
    X_scaled_df
    .assign(
        cluster=cluster_labels
    )
    .groupby("cluster")
    .mean()
)

#نمایش Transpose:
cluster_profile_scaled.T.round(3)


"""
نحوه خواندن Z-score
برای هر Feature:

مقدار مثبت:
بالاتر از میانگین کل Player–Matchها

مقدار منفی:
پایین‌تر از میانگین کل Player–Matchها

نزدیک صفر:
تقریباً مشابه میانگین

shots_per90 = +1.31
یعنی شوت در آن Cluster حدود ۱.۳ انحراف معیار بالاتر از میانگین است.

"""



In [ ]:
##  پروفایل در Model Space
# این جدول از داده Clipped و Imputed ساخته می‌شود؛ یعنی دقیقاً داده‌ای که مدل دیده است.

model_data_clipped = X_scaled_df.copy()
cluster_profile_model_space = (
    model_data_clipped
    .assign(
        cluster=cluster_labels
    )
    .groupby("cluster")
    .mean()
    .round(3)
)

cluster_profile_model_space.T




In [ ]:
## پروفایل خام و قابل‌فهم فوتبالی

cluster_profile_raw = (
    player_match_clustered
    .groupby("cluster")[
        CLUSTER_FEATURES
    ]
    .mean()
    .round(3)
)
cluster_profile_raw.T

In [ ]:
#Median هر Feature در هر Cluster
# چون بعضی Featureها مثل شوت و Box Action توزیع Skewed دارند، فقط میانگین کافی نیست.

cluster_profile_median = (
    player_match_clustered
    .groupby("cluster")[
        CLUSTER_FEATURES
    ]
    .median()
    .round(3)
)
cluster_profile_median.T

In [ ]:
## استخراج قوی‌ترین و ضعیف‌ترین Featureهای هر Cluster

cluster_characteristics = {}
for cluster_id in (cluster_profile_scaled.index):
    profile = (cluster_profile_scaled.loc[cluster_id].sort_values(ascending=False))

    cluster_characteristics[
        cluster_id
    ] = {
        "strongest_features":
            profile.head(6),

        "weakest_features":
            profile.tail(6),
    }

## namayesh:
for (
    cluster_id,
    characteristics,
) in cluster_characteristics.items():

    print("\n" + "=" * 60)
    print(f"CLUSTER {cluster_id}")

    print("\nStrongest features:")
    print(
        characteristics[
            "strongest_features"
        ]
    )

    print("\nWeakest features:")
    print(
        characteristics[
            "weakest_features"
        ]
    )

In [ ]:
## رسم Heatmap پروفایل Clusterها

plt.figure(figsize=(15, 6))

plt.imshow(
    cluster_profile_scaled,
    aspect="auto",
)

plt.colorbar(
    label="Mean standardized value"
)

plt.xticks(
    range(len(CLUSTER_FEATURES)),
    CLUSTER_FEATURES,
    rotation=90,
)

plt.yticks(
    range(FINAL_K),
    [
        f"Cluster {cluster_id}"
        for cluster_id
        in cluster_profile_scaled.index
    ],
)

plt.xlabel("Performance Features")
plt.ylabel("Cluster")

plt.title(
    "Standardized Player-Match "
    "Performance Profiles"
)

plt.tight_layout()
plt.show()

In [ ]:
## PCA Scatter Plot نهایی

plt.figure(figsize=(10, 7))

scatter = plt.scatter(
    player_match_clustered[
        "pca_1"
    ],
    player_match_clustered[
        "pca_2"
    ],
    c=player_match_clustered[
        "cluster"
    ],
    alpha=0.75,
)

plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")

plt.title(
    "SC Freiburg Player-Match "
    "Performance Clusters"
)

plt.colorbar(
    scatter,
    label="Cluster",
)

plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

"""
* Silhouette پایین تا متوسط است
* Player Performance ماهیت پیوسته دارد
* PCA دوبعدی فقط بخشی از کل اطلاعات را نمایش می‌دهد
"""

In [ ]:
## محاسبه Centroidها در فضای PCA
# مرکز هر Cluster را روی نمودار مشخص کنیم:

pca_centroids = (
    player_match_clustered
    .groupby("cluster")[
        [
            "pca_1",
            "pca_2",
        ]
    ]
    .mean()
)


plt.figure(figsize=(10, 7))

scatter = plt.scatter(
    player_match_clustered["pca_1"],
    player_match_clustered["pca_2"],
    c=player_match_clustered["cluster"],
    alpha=0.65,
)

plt.scatter(
    pca_centroids["pca_1"],
    pca_centroids["pca_2"],
    marker="X",
    s=250,
    edgecolors="black",
    linewidths=1.5,
)

for cluster_id, row in (
    pca_centroids.iterrows()
):
    plt.annotate(
        f"Cluster {cluster_id}",
        (
            row["pca_1"],
            row["pca_2"],
        ),
        xytext=(7, 7),
        textcoords="offset points",
    )

plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")

plt.title(
    "Player-Match Performance "
    "Clusters with PCA Centroids"
)

plt.colorbar(
    scatter,
    label="Cluster",
)

plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
### MOHEMM بررسی توزیع پست‌ها در Clusterها

position_cluster_counts = pd.crosstab(
    player_match_clustered[
        "cluster"
    ],
    player_match_clustered[
        "player_position"
    ],
)
position_cluster_counts



In [ ]:
## darsade tozi

position_cluster_percentages = (
    pd.crosstab(
        player_match_clustered[
            "cluster"
        ],
        player_match_clustered[
            "player_position"
        ],
        normalize="index",
    )
    * 100
)
position_cluster_percentages.round(1)

In [ ]:
## بررسی تعداد Cluster برای هر پست

cluster_distribution_by_position = (
    pd.crosstab(
        player_match_clustered[
            "player_position"
        ],
        player_match_clustered[
            "cluster"
        ],
        normalize="index",
    )
    * 100
)

cluster_distribution_by_position.round(1)

"""
مثلاً اگر Center Forwardها در Clusterهای 0 و 2 دیده شوند، نشان می‌دهد مهاجم می‌تواند عملکرد هجومی قوی یا عملکرد کم‌درگیری داشته باشد.
"""


In [ ]:
## بررسی Variation هر بازیکن

player_cluster_variation = (
    player_match_clustered
    .groupby("player_name")
    ["cluster"]
    .agg(
        number_of_clusters="nunique",
        number_of_matches="count",
    )
    .sort_values(
        [
            "number_of_clusters",
            "number_of_matches",
        ],
        ascending=False,
    )
)

player_cluster_variation

In [ ]:
## بازیکنان با بیش از یک نوع Performance:
multi_profile_players = (
    player_cluster_variation[
        player_cluster_variation[
            "number_of_clusters"
        ] > 1
    ]
)
multi_profile_players

In [ ]:
## توزیع Cluster هر بازیکن
# برای Streamlit و تحلیل فردی:

player_cluster_counts = pd.crosstab(
    player_match_clustered[
        "player_name"
    ],
    player_match_clustered[
        "cluster"
    ],
)
player_cluster_counts



In [ ]:
## نسخه درصدی:

player_cluster_percentages = (
    pd.crosstab(
        player_match_clustered[
            "player_name"
        ],
        player_match_clustered[
            "cluster"
        ],
        normalize="index",
    )
    * 100
)
player_cluster_percentages.round(1)
# این جدول می‌تواند بعداً در Player Explorer نشان دهد هر بازیکن چند درصد مسابقاتش را با هر Performance Type انجام داده است.

In [117]:
##  فاصله هر نمونه تا Centroid خودش
# برای پیدا کردن نماینده‌ترین Player–Matchهای هر Cluster:

distance_to_all_centroids = (
    final_kmeans.transform(
        X_scaled_df
    )
)
assigned_centroid_distance = (
    distance_to_all_centroids[
        np.arange(
            len(cluster_labels)
        ),
        cluster_labels,
    ]
)

## اضافه‌کردن به جدول:
player_match_clustered[
    "distance_to_centroid"
] = (
    assigned_centroid_distance
)

In [ ]:
## نماینده‌ترین Performanceها در هر Cluster

representative_performances = (
    player_match_clustered[
        [
            "cluster",
            "match_id",
            "player_id",
            "player_name",
            "player_position",
            "minutes",
            "distance_to_centroid",
            "silhouette_value",
        ]
    ]
    .sort_values(
        [
            "cluster",
            "distance_to_centroid",
        ]
    )
    .groupby(
        "cluster",
        group_keys=False,
    )
    .head(10)
)
representative_performances

In [ ]:
## قوی‌ترین نمونه‌های هر Cluster از نظر عضویت
# نمونه‌هایی با Silhouette بالاتر، واضح‌تر متعلق به Cluster خود هستند:

most_confident_performances = (
    player_match_clustered[
        [
            "cluster",
            "match_id",
            "player_name",
            "player_position",
            "minutes",
            "silhouette_value",
            "distance_to_centroid",
        ]
    ]
    .sort_values(
        [
            "cluster",
            "silhouette_value",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .groupby(
        "cluster",
        group_keys=False,
    )
    .head(10)
)

most_confident_performances

In [ ]:
## بررسی نمونه‌های مرزی
## نمونه‌هایی با Silhouette منفی یا نزدیک صفر:

borderline_performances = (
    player_match_clustered[
        player_match_clustered[
            "silhouette_value"
        ] < 0
    ][
        [
            "cluster",
            "match_id",
            "player_name",
            "player_position",
            "minutes",
            "silhouette_value",
        ]
    ]
    .sort_values(
        "silhouette_value"
    )
)

borderline_performances.head(30)

In [121]:
## بررسی هر Cluster با مقادیر خام

from IPython.display import display

def inspect_cluster(
    cluster_id,
    top_n=10,
):
    print("=" * 70)
    print(f"CLUSTER {cluster_id}")

    print("\nSize:")
    print(
        cluster_size_report
        .loc[cluster_id]
    )

    print("\nStandardized profile:")
    print(
        cluster_profile_scaled
        .loc[cluster_id]
        .sort_values(
            ascending=False
        )
    )

    print("\nRaw mean profile:")
    print(
        cluster_profile_raw
        .loc[cluster_id]
        .sort_values(
            ascending=False
        )
    )

    print("\nPosition distribution (%):")
    print(
        position_cluster_percentages
        .loc[cluster_id]
        .sort_values(
            ascending=False
        )
    )

    print("\nRepresentative performances:")
    display(
        representative_performances[
            representative_performances[
                "cluster"
            ] == cluster_id
        ].head(top_n)
    )

    inspect_cluster(0)
    #inspect_cluster(1)
    #inspect_cluster(2)

In [ ]:
cluster_profile_scaled.T.round(2)

In [ ]:

## naming the cluster

CLUSTER_NAME_MAP = {
    0: "Defensive Performance",
    1: "High-Impact Attacking Performance",
    2: "Support Performance",
}

player_match_clustered["cluster_name"] = (
    player_match_clustered["cluster"]
    .map(CLUSTER_NAME_MAP)
)

player_match_clustered["cluster_name"]

In [124]:
## performance dimensions summary

CLUSTER_DESCRIPTION_MAP = {
    2: (
        "Lower involvement with "
        "a support of  wide-availability role."
    ),

    0: (
        "High defensive contribution and "
        "involvement, with strong "
        "ball recovery and passing activity."
    ),

    1: (
        "High attacking threat through shots, "
        "box involvement, final third actions, "
        "and possession threat."
    ),
}

player_match_clustered[
    "cluster_description"
] = (
    player_match_clustered[
        "cluster"
    ]
    .map(
        CLUSTER_DESCRIPTION_MAP
    )
)

In [ ]:
## جدول خلاصه نهایی Clusterها
cluster_summary_table = (
    cluster_size_report
    .reset_index()
    .rename(
        columns={
            "index": "cluster"
        }
    )
)

cluster_summary_table[
    "cluster_name"
] = (
    cluster_summary_table[
        "cluster"
    ]
    .map(CLUSTER_NAME_MAP)
)
## silhoutte
cluster_summary_table = (
    cluster_summary_table
    .merge(
        cluster_silhouette_report
        .reset_index(),
        on="cluster",
        how="left",
    )
)

cluster_summary_table = (
    cluster_summary_table[
        [
            "cluster",
            "cluster_name",
            "count",
            "percentage",
            "mean_silhouette",
            "median_silhouette",
            "negative_percentage",
        ]
    ]
)

cluster_summary_table.round(3)

In [ ]:
## ذخیره Figureها
from pathlib import Path

OUTPUT_DIR = Path("../outputs")

FIGURE_DIR = OUTPUT_DIR / "figures"

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

FIGURE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# ذخیره PCA Scatter
plt.figure(figsize=(10, 7))

scatter = plt.scatter(
    player_match_clustered["pca_1"],
    player_match_clustered["pca_2"],
    c=player_match_clustered["cluster"],
    alpha=0.75,
)

plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")

plt.title(
    "SC Freiburg Player-Match "
    "Performance Clusters"
)

plt.colorbar(
    scatter,
    label="Cluster",
)

plt.grid(alpha=0.2)
plt.tight_layout()

plt.savefig(
    FIGURE_DIR
    / "player_match_clusters_pca.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

## ذخیره Heatmap

plt.figure(figsize=(15, 6))

plt.imshow(
    cluster_profile_scaled,
    aspect="auto",
)

plt.colorbar(
    label="Mean standardized value"
)

plt.xticks(
    range(len(CLUSTER_FEATURES)),
    CLUSTER_FEATURES,
    rotation=90,
)

plt.yticks(
    range(FINAL_K),
    [
        CLUSTER_NAME_MAP[
            cluster_id
        ]
        for cluster_id
        in cluster_profile_scaled.index
    ],
)

plt.title(
    "Standardized Performance "
    "Profile by Cluster"
)

plt.tight_layout()

plt.savefig(
    FIGURE_DIR
    / "cluster_profile_heatmap.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

In [127]:
## ذخیره Datasetهای نهایی

player_match_clustered.to_csv(
    OUTPUT_DIR
    / "player_match_clusters.csv",
    index=False,
)

##  گزارش اندازه و کیفیت Clusterها
cluster_summary_table.to_csv(
    OUTPUT_DIR
    / "cluster_summary.csv",
    index=False,
)

In [128]:
## 31  ذخیره Feature List
## for reproducibility:

import json
with open(
    OUTPUT_DIR
    / "cluster_features.json",
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        CLUSTER_FEATURES,
        file,
        indent=4,
    )

    with open(
    OUTPUT_DIR
    / "cluster_name_map.json",
    "w",
    encoding="utf-8",
) as file:
        json.dump(
    CLUSTER_NAME_MAP,
        file,
        indent=4,
)

In [ ]:
## ذخیره مدل‌ها برای استفاده در Streamlit:
import joblib

#ذخیره K-Means:

joblib.dump(
    final_kmeans,
    OUTPUT_DIR
    / "final_kmeans_model.joblib",
)

# ذخیره Scaler:

joblib.dump(
    scaler,
    OUTPUT_DIR
    / "standard_scaler.joblib",
)

## ذخیره PCA دوبعدی:

joblib.dump(
    pca_2d,
    OUTPUT_DIR
    / "pca_2d_model.joblib",
)

## اگر Medianها را ذخیره کرده‌ای:
joblib.dump(
    feature_medians,
    OUTPUT_DIR
    / "feature_medians.joblib",
)

## ذخیره Clipping Boundها:

joblib.dump(
    clipping_bounds,
    OUTPUT_DIR
    / "clipping_bounds.joblib",
)

In [ ]:
## تست Load شدن مدل‌ها

loaded_kmeans = joblib.load(
    OUTPUT_DIR
    / "final_kmeans_model.joblib"
)

loaded_scaler = joblib.load(
    OUTPUT_DIR
    / "standard_scaler.joblib"
)


#Prediction مجدد:
loaded_labels = (
    loaded_kmeans.predict(
        X_scaled_df
    )
)

# بررسی:
assert np.array_equal(
    loaded_labels,
    cluster_labels,
)

print(
    "Saved model reproduces "
    "the same cluster labels."
)

In [ ]:
## کنترل نهایی کل Pipeline

print("=" * 70)
print("FINAL CLUSTERING CHECK")
print("=" * 70)

print(
    "Player-match samples:",
    len(player_match_clustered),
)

print(
    "Unique players:",
    player_match_clustered[
        "player_id"
    ].nunique(),
)

print(
    "Unique matches:",
    player_match_clustered[
        "match_id"
    ].nunique(),
)

print(
    "Number of features:",
    len(CLUSTER_FEATURES),
)

print(
    "Number of clusters:",
    player_match_clustered[
        "cluster"
    ].nunique(),
)

print(
    "Missing cluster labels:",
    player_match_clustered[
        "cluster"
    ].isna().sum(),
)

print(
    "Missing cluster names:",
    player_match_clustered[
        "cluster_name"
    ].isna().sum(),
)

print(
    "Duplicate player-match rows:",
    player_match_clustered
    .duplicated(
        subset=[
            "match_id",
            "player_id",
        ]
    )
    .sum(),
)

print(
    "Final silhouette:",
    round(
        final_silhouette,
        4,
    ),
)

print("\nCluster sizes:")
print(cluster_size_report.round(2))

In [ ]:
sorted(
    [
        path.name
        for path in OUTPUT_DIR.iterdir()
    ]
)